In [10]:
from repository_knowledge_assistant.ingestion.clone import RepositoryCloner
from repository_knowledge_assistant.ingestion.load import RepositoryLoader
from repository_knowledge_assistant.ingestion.parse import RepositoryParser
from repository_knowledge_assistant.ingestion.chunk import RepositoryChunker
from repository_knowledge_assistant.ingestion.embed import RepositoryEmbedder
from repository_knowledge_assistant.search.elasticsearch import Index
url = "https://github.com/pypa/sampleproject.git" 
repo = RepositoryCloner().get_repository(url)
docs = RepositoryLoader().load(repo)
parser = RepositoryParser()
chunker = RepositoryChunker()
chunks = []
for doc in docs:
    docs2 = parser.parse(doc)
    for doc2 in docs2:
        chunks.append(chunker.chunk(doc2))
chunks = sum(chunks, [])

embed = RepositoryEmbedder()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
embeddings = embed.embed_documents(chunks)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [2]:
index = Index(index_name="sample_project")
index.create_index()

In [5]:
len(embeddings)

15

In [4]:
index.index_documents(embeddings)

In [7]:
chunks[0].chunk_id

'r0_d0_c0'

In [ ]:
index.keyword_search(['def'])


[{'id': 'r29_d0_c0',
  'score': 1.4688402,
  'path': 'data/repos/sampleproject/src/sample/simple.py',
  'name': 'simple.py',
  'text': 'def add_one(number):\n    return number + 1'},
 {'id': 'r11_d0_c0',
  'score': 1.4339924,
  'path': 'data/repos/sampleproject/tests/test_simple.py',
  'name': 'test_simple.py',
  'text': 'class TestSimple(unittest.TestCase):\n\n    def test_add_one(self):\n        self.assertEqual(add_one(5), 6)'},
 {'id': 'r28_d0_c0',
  'score': 1.3926907,
  'path': 'data/repos/sampleproject/src/sample/__init__.py',
  'name': '__init__.py',
  'text': 'def main():\n    """Entry point for the application script"""\n    print("Call your main application code here")'},
 {'id': 'r4_d0_c0',
  'score': 1.3847142,
  'path': 'data/repos/sampleproject/noxfile.py',
  'name': 'noxfile.py',
  'text': 'def lint(session):\n    session.install("flake8")\n    session.run(\n        "flake8", "--exclude", ".nox,*.egg,build,data",\n        "--select", "E,W,F", "."\n    )'},
 {'id': 'r4_d

In [9]:
len(index.client.search(index=index.index_name, query={"match_all": {}}, size=20)['hits']['hits'])

15

In [17]:
index.client.count(index=index.index_name)

ObjectApiResponse({'count': 6, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

In [9]:
print("Elementos:", len(embeddings))
print("IDs únicos:", len(set(chunk.chunk_id for chunk in chunks)))

Elementos: 15
IDs únicos: 6


In [3]:
for e in embeddings:
    print(f"{e.chunk_id}: {e.text[:20]}")

r0_d0_c0: Copyright (c) 2016 T
r3_d1_c0: Section: A sample Py
r4_d0_c0: def lint(session):
 
r4_d1_c0: def build_and_check_
r4_d2_c0: def tests(session):

r4_d3_c0: import os
import nox
r8_d0_c0: # Guide (user-friend
r8_d0_c1: (...)

# This is eit
r8_d0_c2: (...)

# List additi
r11_d0_c0: class TestSimple(uni
r11_d1_c0: import unittest
from
r25_d0_c0: # this file is *not*
r26_d0_c0: # this file is *not*
r28_d0_c0: def main():
    """E
r29_d0_c0: def add_one(number):
